# Investigando CVLI no Ceará (2009–2025): Uma Análise Exploratória e Regional
## Relatório Técnico-Analítico Fundacional

**Objetivo Central:** Construir uma base diagnóstica rigorosa que descreva os padrões, a evolução temporal, a dinâmica espacial e o perfil demográfico dos Crimes Violentos Letais Intencionais (CVLI) no estado do Ceará.

**Variável Target Futura:** `total_cvli_ano` — contagem de CVLI por município/ano  
**Unidade Analítica Complementar:** As 14 Regiões de Planejamento do Ceará  
**Estrutura:** Módulos customizados em `src/` e base de dados tratada em `data/`

---

## Seção 0 — Qual é a estrutura dos dados e como configurar o ambiente de análise?

### 0.1 Instalação e verificação de dependências

In [ ]:
!pip install geobr geopandas matplotlib seaborn pandas openpyxl

### 0.2 Importação dos módulos do projeto (src) e pacotes auxiliares

In [ ]:
import sys
import os

# Garantir importação dos módulos locais de src/
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('.'))

import pandas as pd
import geobr as gpd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data import (
    load_cvli_data,
    load_planning_regions,
    load_municipality_geodata,
    clean_column_names,
    merge_geodata_and_regions
)
from src.features import (
    extract_temporal_features,
    create_age_groups,
    categorize_rmf_vs_interior,
    calculate_regional_metrics
)
from src.visualization import (
    set_chart_style,
    plot_temporal_trend,
    plot_top_municipalities,
    plot_planning_regions,
    plot_regional_facet_trends,
    plot_rmf_vs_interior_trend,
    plot_crime_nature_breakdown,
    plot_victim_demographics,
    plot_monthly_heatmap,
    plot_seasonality_bars
)

# Ativar configuração estética padronizada
set_chart_style()
print("Módulos de src/ carregados com sucesso!")


### 0.3 Carregamento, limpeza inicial e unificação espacial

In [ ]:
# 1. Carregar dados brutos
df_raw = load_cvli_data()
planejamento = load_planning_regions()
geo_ce = load_municipality_geodata()

# 2. Padronizar nomes das colunas
df = clean_column_names(df_raw)

# 3. Vincular espacialmente microdados e regiões de planejamento (Evita bugs de coluna ausente)
df = merge_geodata_and_regions(df, geo_ce, planejamento)

# 4. Criar o painel agregado por município e ano (Target: total_cvli_ano)
agregado = df.groupby(['município', 'ano']).size().reset_index(name='total_cvli_ano')
agregado = merge_geodata_and_regions(agregado, geo_ce, planejamento)

print(f"Microdados de CVLI: {df.shape[0]:,} ocorrências × {df.shape[1]} colunas")
print(f"Painel Agregado: {agregado.shape[0]:,} linhas (município-ano) em {agregado['município'].nunique()} municípios")
display(df.head())


---
## Seção 1 — Qual é o diagnóstico de qualidade, integridade e ausência nos dados?

In [ ]:
print("=" * 65)
print("DIAGNÓSTICO DE INTEGRIDADE E VALORES AUSENTES (SKILL ANÁLISE)")
print("=" * 65)

# Diagnóstico de NaNs
missing = df[['município', 'natureza', 'data', 'idade_da_vítima', 'escolaridade_da_vítima', 'raça_da_vítima', 'ano']].isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
display(missing.sort_values("missing_pct", ascending=False))

# Diagnóstico de 'Não Informada'
print("\n" + "=" * 65)
print("VALORES 'NÃO INFORMADA' EM ATRIBUTOS DEMOGRÁFICOS")
print("=" * 65)

for col in ['idade_da_vítima', 'escolaridade_da_vítima', 'raça_da_vítima']:
    if col in df.columns:
        nao_inf = (df[col] == 'Não Informada').sum()
        pct = (nao_inf / len(df) * 100).round(2)
        print(f"  Column '{col}': {nao_inf:,} ausências registradas ({pct}%)")

print('''
Diagnóstico Técnico:
- Raça da Vítima: ~77% de ausência estrutural ('Não Informada').
- Escolaridade: Alta frequência de campos não preenchidos.
- Idade da Vítima: Subconjunto numérico válido precisa ser filtrado para estatísticas.

Diretriz Metodológica:
As análises demográficas utilizarão estritamente o subconjunto com dados informados.
''')


---
## Seção 2 — Como a violência letal evoluiu no Ceará entre 2009 e 2025?

In [ ]:
# Preparar dados agregados estaduais por ano
df_temporal = df.groupby('ano').size().reset_index(name='total_cvli')
df_temporal['variacao_pct'] = (df_temporal['total_cvli'].pct_change() * 100).round(2)

print("Tabela-resumo: Evolução Anual do CVLI no Ceará:")
display(df_temporal)

# Gerar gráfico de evolução temporal refinado
fig, (ax1, ax2) = plot_temporal_trend(df_temporal)
plt.show()

print('''
SÍNTESE ANALÍTICA:
- Fato Observado: O Ceará registrou oscilações marcantes na série histórica (2009-2025).
- Evidência: O pico atingiu mais de 5.000 homicídios nos anos de crise de segurança (2017-2018).
- Hipótese: Mudanças na dinâmica de facções criminosas e reformas no policiamento impactaram a trajetória.
- Limitação: Análise agregada estadual oculta discrepâncias entre regiões.
''')


---
## Seção 3 — Em quais municípios e regiões de planejamento a violência mais se concentra?

### 3.1 Qual é o ranking dos municípios mais violentos?

In [ ]:
ranking_muni = df.groupby('município').size().reset_index(name='total_cvli')
ranking_muni = ranking_muni.sort_values('total_cvli', ascending=False).reset_index(drop=True)
ranking_muni['pct_total'] = (ranking_muni['total_cvli'] / ranking_muni['total_cvli'].sum() * 100).round(2)
ranking_muni['pct_acumulado'] = ranking_muni['pct_total'].cumsum().round(2)

display(ranking_muni.head(20))

fig, ax = plot_top_municipalities(ranking_muni, top_n=20)
plt.show()


### 3.2 Como o volume de homicídios se distribui entre as 14 Regiões de Planejamento?

In [ ]:
# Métricas por Região de Planejamento
ranking_regiao = calculate_regional_metrics(df, region_column='regiao_planejamento')
print("Tabela Comparativa por Região de Planejamento:")
display(ranking_regiao)

# Gráfico comparativo regional
fig, ax = plot_planning_regions(ranking_regiao)
plt.show()


### 3.3 Qual a trajetória histórica individual de cada uma das 14 Regiões de Planejamento?

In [ ]:
# Agregação anual por região de planejamento
evolucao_regiao = agregado.groupby(['regiao_planejamento', 'ano'])['total_cvli_ano'].sum().reset_index()

# FacetGrid de todas as 14 regiões
g = plot_regional_facet_trends(evolucao_regiao)
plt.show()


---
## Seção 4 — Qual é a composição penal/jurídica dos crimes violentos no estado?

In [ ]:
dist_natureza = df['natureza'].value_counts().reset_index()
dist_natureza.columns = ['natureza', 'total']
dist_natureza['pct'] = (dist_natureza['total'] / dist_natureza['total'].sum() * 100).round(2)

evo_natureza = df.groupby(['ano', 'natureza']).size().reset_index(name='total')

display(dist_natureza)

fig, (ax1, ax2) = plot_crime_nature_breakdown(dist_natureza, evo_natureza)
plt.show()


---
## Seção 5 — Qual é o perfil demográfico das vítimas com registros disponíveis?

In [ ]:
# Preparar featurização demográfica
df_feat = create_age_groups(df)

# Faixa Etária
dist_idade = df_feat['faixa_etaria'].value_counts().sort_index().reset_index()
dist_idade.columns = ['faixa_etaria', 'total']
dist_idade['pct'] = (dist_idade['total'] / dist_idade['total'].sum() * 100).round(2)

# Escolaridade
df_esc = df[df['escolaridade_da_vítima'] != 'Não Informada']
dist_esc = df_esc['escolaridade_da_vítima'].value_counts().reset_index()
dist_esc.columns = ['escolaridade', 'total']
dist_esc['pct'] = (dist_esc['total'] / dist_esc['total'].sum() * 100).round(2)

# Raça
df_raca = df[df['raça_da_vítima'] != 'Não Informada']
dist_raca = df_raca['raça_da_vítima'].value_counts().reset_index()
dist_raca.columns = ['raca', 'total']
dist_raca['pct'] = (dist_raca['total'] / dist_raca['total'].sum() * 100).round(2)

fig, axes = plot_victim_demographics(dist_idade, dist_esc, dist_raca)
plt.show()


---
## Seção 6 — Como a dinâmica da violência difere entre a Região Metropolitana de Fortaleza e o Interior?

In [ ]:
# Categorizar RMF vs Interior tanto no painel quanto nos microdados (sem bugs de merge)
agregado_grupo = categorize_rmf_vs_interior(agregado)
df_grupo = categorize_rmf_vs_interior(df)

comp_grupo = agregado_grupo.groupby('grupo')['total_cvli_ano'].sum().reset_index()
comp_grupo['pct'] = (comp_grupo['total_cvli_ano'] / comp_grupo['total_cvli_ano'].sum() * 100).round(2)
print("Resumo Comparativo RMF vs Interior:")
display(comp_grupo)

# Evolução temporal comparativa RMF vs Interior
evo_grupo = agregado_grupo.groupby(['ano', 'grupo'])['total_cvli_ano'].sum().reset_index()
evo_grupo.rename(columns={'total_cvli_ano': 'total_cvli'}, inplace=True)

fig, ax = plot_rmf_vs_interior_trend(evo_grupo)
plt.show()

# Comparação do município líder de cada uma das 14 Regiões de Planejamento
print("\nMunicípio líder em homicídios de cada Região de Planejamento:")
lideres_regionais = (
    df.groupby(['regiao_planejamento', 'município'])
    .size()
    .reset_index(name='total_cvli')
    .sort_values(['regiao_planejamento', 'total_cvli'], ascending=[True, False])
    .groupby('regiao_planejamento')
    .first()
    .reset_index()
)
display(lideres_regionais)


---
## Seção 7 — Existem padrões sazonais na ocorrência dos homicídios por mês ou dia da semana?

In [ ]:
df_temp_feat = extract_temporal_features(df)

# Matrix Mês x Ano para o Heatmap
heatmap_data = df_temp_feat.groupby(['ano', 'mes']).size().reset_index(name='total')
heatmap_pivot = heatmap_data.pivot(index='mes', columns='ano', values='total').fillna(0)

meses_nome = {1:'Jan', 2:'Fev', 3:'Mar', 4:'Abr', 5:'Mai', 6:'Jun',
              7:'Jul', 8:'Ago', 9:'Set', 10:'Out', 11:'Nov', 12:'Dez'}
heatmap_pivot.index = [meses_nome[m] for m in heatmap_pivot.index if m in meses_nome]

# Heatmap Mês x Ano
fig, ax = plot_monthly_heatmap(heatmap_pivot)
plt.show()

# Distribuição Mês e Dia da Semana
dist_mes = df_temp_feat.groupby('mes').size().reset_index(name='total')
dist_mes['mes_nome'] = dist_mes['mes'].map(meses_nome)

dist_dia = df_temp_feat.groupby('dia_semana').size().reset_index(name='total')
dias_nome = {0:'Seg', 1:'Ter', 2:'Qua', 3:'Qui', 4:'Sex', 5:'Sáb', 6:'Dom'}
dist_dia['dia_nome'] = dist_dia['dia_semana'].map(dias_nome)

fig, axes = plot_seasonality_bars(dist_mes, dist_dia)
plt.show()


---
## Seção 8 — Quais evidências foram consolidadas e como elas estruturam a futura modelagem espacial e de ML?

In [ ]:
print("=" * 70)
print("SÍNTESE DIAGNÓSTICA E MAPA PARA MODELAGEM ESPACIAL / MACHINE LEARNING")
print("=" * 70)

print(f'''
1. BASE DE DADOS TRATADA E SANITIZADA
   - Registros totais: {len(df):,} ocorrências no Ceará (2009-2025)
   - Painel Municipal Agregado: {len(agregado):,} observações (184 municípios x 17 anos)
   - Integração com as 14 Regiões de Planejamento concluída sem perdas.

2. ACHADOS REGIONAIS E ESPACIAIS CHAVE
   - Elevada concentração: Grande Fortaleza responde por ~50% do total de CVLI do estado.
   - Disparidade Regional: Regiões como Cariri e Sobral destacam-se como polos secundários de atenção.
   - Sazonalidade: Elevação estatisticamente relevante nos fins de semana (Sábado/Domingo).

3. ESTRUTURA PARA MODELAGEM FUTURA (TARGET & PREDICTORS)
   - Variable Target: total_cvli_ano (Contagem de homicídios no painel municipal)
   - Próximo Passo 1: Incorporar população municipal IBGE para calcular Taxa por 100k hab.
   - Próximo Passo 2: Construir Matriz de Ponderação Espacial (W Matrix) para econometria (SAR/SEM/GWR).
   - Próximo Passo 3: Treinar modelos preditivos via scikit-learn/XGBoost usando os módulos de src/models.
''')
